# Prompt evaluation

## Generating test datasets

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system = None, temperature = 1.0, stop_sequences = []):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [9]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences = ["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a regular expression to validate AWS S3 bucket names. S3 bucket names must be between 3 and 63 characters long, contain only lowercase letters, numbers, hyphens, and periods, start and end with a letter or number, and cannot contain consecutive hyphens or periods.'}, {'task': 'Write a Python function that takes an AWS CloudWatch log event JSON object and extracts the timestamp, log level, and message fields. Return them as a dictionary.'}, {'task': "Write a JSON object that represents an AWS IAM policy allowing a user to read and list objects in a specific S3 bucket named 'my-company-data' but deny access to objects with the prefix 'confidential/'."}]


In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 4)

## Running the eval

In [3]:
def run_prompt(test_case):
    # Merges the prompt and test case input, then returns the result
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [4]:
def run_test_case(test_case):
    # Calls run_prompt, then grades the result
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [5]:
def run_eval(dataset):
    # Loads the dataset and calls run_test_case with each case
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [ ]:
with open(r"/../datasets/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

FileNotFoundError: [Errno 2] No such file or directory: '/../datasets/dataset.json'

In [15]:
from pathlib import Path
print(Path.cwd())

c:\Users\LeonardoMuñoz\OneDrive - SPS\Documentos\Git\building-with-the-claude-api\jupyter_notebooks\prompt_evaluation


In [ ]:
with open(r"../../datasets/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [28]:
print(json.dumps(results, indent = 4))

[
    {
        "output": "# Solution\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Examples:\n        - 's3://my-bucket.s3.us-west-2.amazonaws.com/key' -> 'us-west-2'\n        - 'https://my-bucket.s3.eu-central-1.amazonaws.com/key' -> 'eu-central-1'\n        - 'https://my-bucket.s3.amazonaws.com/key' -> None (us-east-1 implicit)\n    \n    Args:\n        url (str): S3 bucket URL\n        \n    Returns:\n        str: AWS region code (e.g., 'us-west-2') or None if region not found\n    \"\"\"\n    # Pattern to match region in S3 URL: .s3.<region>.amazonaws.com\n    pattern = r'\\.s3\\.([a-z0-9\\-]+)\\.amazonaws\\.com'\n    match = re.search(pattern, url)\n    \n    if match:\n        return match.group(1)\n    return None\n\n\n# Test cases\nif __name__ == \"__main__\":\n    test_urls = [\n        's3://my-bucket.s3.u

## Model based grading

In [6]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>
    
    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields, in this specific order:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your overall assessment
    - "score": A number between 1-10
    
    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences = ["```"])
    return json.loads(eval_text)

In [7]:
def run_test_case(test_case):
    # Calls run_prompt, then grades the result
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [10]:
with open(r"../../datasets/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent = 4))

[
    {
        "output": "# AWS Region Extraction from S3 Bucket URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        url: S3 bucket URL in the format:\n             - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n             - https://my-bucket.s3.us-west-2.amazonaws.com/key\n             - https://my-bucket.s3.amazonaws.com/key (returns 'us-east-1')\n    \n    Returns:\n        The AWS region (e.g., 'us-west-2'), or 'us-east-1' if not specified.\n    \n    Raises:\n        ValueError: If the URL format is invalid.\n    \"\"\"\n    # Pattern to match S3 URLs with region\n    pattern = r's3://[^.]+\\.s3\\.([a-z0-9\\-]+)\\.amazonaws\\.com|https?://[^.]+\\.s3\\.([a-z0-9\\-]+)\\.amazonaws\\.com|https?://[^.]+\\.s3\\.amazonaws\\.com'\n    \n    match = re.search(pattern, url)\n    \n    if not

In [11]:
from statistics import mean

def run_eval(dataset):
    # Loads the dataset and calls run_test_case with each case
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [12]:
with open(r"../../datasets/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent = 4))

Average score: 6
[
    {
        "output": "# Solution: Extract AWS Region from S3 Bucket URL\n\nHere's a Python function that extracts the AWS region from an S3 bucket URL:\n\n```python\nimport re\n\ndef extract_region_from_s3_url(s3_url: str) -> str:\n    \"\"\"\n    Extracts the AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url: S3 URL in formats like:\n                - s3://my-bucket.s3.us-west-2.amazonaws.com/key\n                - https://my-bucket.s3.us-west-2.amazonaws.com/key\n                - s3://bucket-name/key (returns None for default region)\n    \n    Returns:\n        The AWS region string (e.g., 'us-west-2') or None if not found\n    \"\"\"\n    # Pattern to match region in S3 URLs\n    # Matches: .s3.REGION.amazonaws.com or .s3-REGION.amazonaws.com\n    pattern = r'\\.s3[.-]([a-z0-9-]+)\\.amazonaws\\.com'\n    \n    match = re.search(pattern, s3_url)\n    if match:\n        return match.group(1)\n    \n    return None\n\n\n# Test cases\nif __name__